In [2]:
!git clone https://github.com/vinhVVN/Realized-Volatility-Prediction.git

Cloning into 'Realized-Volatility-Prediction'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 121 (delta 46), reused 102 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 76.83 KiB | 6.98 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [1]:
import os
import sys
import gc
import glob
import joblib
import numpy as np
import pandas as pd
import warnings
import torch
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

In [3]:
PROJECT_ROOT = '/kaggle/working/Realized-Volatility-Prediction'
if os.path.exists(PROJECT_ROOT): os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)

In [4]:
from src.data.data_loader import DataBlock
from src.features.feature_pipeline import build_features
from src.models.cnn_model import CNN1DModel
from src.models.mlp_model import MLPModel

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Bắt đầu quá trình Inference trên: {device}")

# 2. 🗂️ ĐỌC DỮ LIỆU KIỂM THỬ (TEST SET) BÍ MẬT CỦA KAGGLE
DATA_DIR = '/kaggle/input/competitions/optiver-realized-volatility-prediction'
test_csv_path = os.path.join(DATA_DIR, 'test.csv')

df_test = pd.read_csv(test_csv_path)

# Tạo row_id chuẩn xác theo format Kaggle yêu cầu
df_test['row_id'] = df_test['stock_id'].astype(str) + '-' + df_test['time_id'].astype(str)
print(f"Kích thước tập Test ban đầu: {df_test.shape}")

🚀 Bắt đầu quá trình Inference trên: cpu
Kích thước tập Test ban đầu: (3, 3)


In [7]:
print("\nĐang khởi động cỗ máy trích xuất K-NN Features cho Test Set...")
df_features = build_features(
    df_train=df_test, 
    data_dir=DATA_DIR, 
    block=DataBlock.TEST, 
    config_path='./configs/feature_config.yaml'
)

# Phân định cột đặc trưng
not_features = ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']
features = [col for col in df_features.columns if col not in not_features]
print(f"Đã trích xuất thành công {len(features)} đặc trưng.")


Đang khởi động cỗ máy trích xuất K-NN Features cho Test Set...
2026-06-09 11:29:00 | INFO     | feature_pipeline | Đã phát hiện 1 unique stock_id trong df_train. Tiến hành filter dữ liệu đọc vào RAM.
2026-06-09 11:29:00 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Bắt đầu...
2026-06-09 11:29:00 | INFO     | data_loader | Đang đọc song song 1 file book test (n_jobs=4)...
2026-06-09 11:29:01 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (3, 11)
2026-06-09 11:29:01 | INFO     | data_loader | Đang đọc song song 1 file trade test (n_jobs=4)...
2026-06-09 11:29:02 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (3, 6)
2026-06-09 11:29:02 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Hoàn thành trong 1.6819 giây.
2026-06-09 11:29:02 | INFO     | feature_pipeline | [b) Tính toán Base Features và Tick Size] Bắt đầu...
2026-06-09 11:29:02 | INFO     | feature_pipeline | [b) Tính toán Base Features và Ti

LinAlgError: 0-dimensional array given. Array must be at least two-dimensional

In [ ]:
# 4. 🧹 TIỀN XỬ LÝ CHO DEEP LEARNING
# Điền NaN bằng 0 (Hoặc mean) để Nơ-ron không bị lỗi "loss: nan"
df_features[features] = df_features[features].replace([np.inf, -np.inf], np.nan)
df_features[features] = df_features[features].fillna(0)

# Chuyển đổi dữ liệu sang Tensor và ép Scaler
X_test_raw = df_features[features].values
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test_raw) # Dùng trực tiếp Scaler trên Test set
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
input_dim = len(features)

In [ ]:
# 5. 🤖 ĐÁNH THỨC CÁC MÔ HÌNH VÀ THỰC THI DỰ ĐOÁN (INFERENCE)
predictions = np.zeros(len(df_features))

# --- A. LIGHTGBM ENSEMBLE ---
lgbm_paths = glob.glob('./models/lgbm/*.pkl')
lgbm_preds = np.zeros(len(df_features))
if lgbm_paths:
    print(f"\n🌳 Đang gọi {len(lgbm_paths)} mô hình LightGBM...")
    for p in lgbm_paths:
        model = joblib.load(p)
        lgbm_preds += model.predict(df_features[features])
    lgbm_preds /= len(lgbm_paths)

# --- B. MLP ENSEMBLE ---
mlp_paths = glob.glob('./models/mlp/*.pth')
mlp_preds = np.zeros(len(df_features))
if mlp_paths:
    print(f"⚡ Đang gọi {len(mlp_paths)} mô hình MLP...")
    for p in mlp_paths:
        model = MLPModel(num_features=input_dim).to(device)
        model.load_state_dict(torch.load(p, map_location=device))
        model.eval()
        with torch.no_grad():
            preds = model(X_test_tensor).cpu().numpy()
            if preds.ndim > 1: preds = preds.flatten() # Đảm bảo mảng 1D
            mlp_preds += preds
    mlp_preds /= len(mlp_paths)

# --- C. 1D-CNN ENSEMBLE ---
cnn_paths = glob.glob('./models/cnn/*.pth')
cnn_preds = np.zeros(len(df_features))
if cnn_paths:
    print(f"🎥 Đang gọi {len(cnn_paths)} mô hình 1D-CNN...")
    for p in cnn_paths:
        model = CNN1DModel(num_features=input_dim).to(device)
        model.load_state_dict(torch.load(p, map_location=device))
        model.eval()
        with torch.no_grad():
            preds = model(X_test_tensor).cpu().numpy()
            if preds.ndim > 1: preds = preds.flatten()
            cnn_preds += preds
    cnn_preds /= len(cnn_paths)

In [ ]:
# 6. ⚖️ TRỘN TRỌNG SỐ THÔNG MINH (SMART WEIGHTED ENSEMBLE)
# Do thực nghiệm của em đã chứng minh LightGBM và MLP cực kỳ mạnh, ta chia tỷ trọng như sau:
W_LGBM = 0.50  # 50% Niềm tin vào Cây quyết định
W_MLP = 0.40   # 40% Niềm tin vào Mạng Nơ-ron MLP
W_CNN = 0.10   # 10% Niềm tin vào Mạng Tích chập CNN để lấy tính đa dạng không gian

print(f"\n⚖️ Áp dụng Trọng số Gộp: LightGBM ({W_LGBM*100}%) | MLP ({W_MLP*100}%) | CNN ({W_CNN*100}%)")

final_preds = (lgbm_preds * W_LGBM) + (mlp_preds * W_MLP) + (cnn_preds * W_CNN)

In [ ]:
# 7. 📤 KẾT XUẤT SUBMISSION
df_features['target'] = final_preds
submission = df_features[['row_id', 'target']]

# Kaggle cần ghi file ra thư mục mặc định `/kaggle/working/submission.csv`
submission.to_csv('/kaggle/working/submission.csv', index=False)


print("✅ HOÀN TẤT DỰ ÁN! File submission.csv đã được sinh ra thành công!")
print(submission.head())